In [1]:

!pip install -U peft transformers torchao bitsandbytes

In [10]:
from google.colab import drive
import yaml, os, torch, glob
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import classification_report
import numpy as np

# 1. Mount Drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/dataMiningProject/CSI_Project"
YAML_PATH = f"{BASE_DIR}/config.yaml"

with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['learning_rate'] = 8e-6
cfg['epochs'] = 30
cfg['rdrop_alpha'] = 1.0
cfg['scl_weight'] = 0.25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Config Loaded. Training {cfg['model_name']} for {cfg['epochs']} epochs.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Config Loaded. Training microsoft/graphcodebert-base for 30 epochs.


In [3]:
class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg['model_name'])

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=cfg['lora_target_modules'],
            bias="none"
        )
        self.encoder = get_peft_model(self.encoder, peft_config)
        self.cwe_head = nn.ModuleDict({
            'norm': nn.LayerNorm(768),
            'fc1': nn.Linear(768, 384),
            'fc2': nn.Linear(384, cfg['num_cwe_classes'])
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        features = outputs.last_hidden_state[:, 0, :]
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        x = self.cwe_head['norm'](features)
        x = self.cwe_head['fc1'](x)
        x = torch.relu(x)
        logits = self.cwe_head['fc2'](x)

        return {"logits": logits, "features": features}

# Loss Functions
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, p=2, dim=1)
        logits = torch.matmul(features, features.T) / self.temperature
        mask = torch.eq(labels.view(-1, 1), labels.view(-1, 1).T).float().to(DEVICE)
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(labels.shape[0]).view(-1, 1).to(DEVICE),
            0
        )
        mask = mask * logits_mask
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach() # Stability trick

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)

        return -mean_log_prob_pos.mean()

class RDropLoss(nn.Module):
    def __init__(self, alpha):
        super().__init__()
        self.alpha = alpha
    def forward(self, l1, l2, target, weights):
        ce = nn.CrossEntropyLoss(weight=weights)
        kl = nn.KLDivLoss(reduction='batchmean')
        ce_loss = 0.5 * (ce(l1, target) + ce(l2, target))
        kl_loss = 0.5 * (kl(F.log_softmax(l1, dim=-1), F.softmax(l2, dim=-1)) +
                         kl(F.log_softmax(l2, dim=-1), F.softmax(l1, dim=-1)))
        return ce_loss + self.alpha * kl_loss

model = GraphCodeBERTLoRACWEModel().to(DEVICE)
scl_criterion = SupervisedContrastiveLoss(cfg['scl_temperature']).to(DEVICE)
rdrop_criterion = RDropLoss(cfg['rdrop_alpha']).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
from torch.utils.data import  WeightedRandomSampler

def get_balanced_sampler(dataset):
    labels = dataset.labels.numpy()
    class_counts = np.bincount(labels, minlength=cfg['num_cwe_classes'])
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler, class_weights


cache_path = f"{BASE_DIR}/{cfg['token_cache_file']}"
if os.path.exists(cache_path):
    cache = torch.load(cache_path, map_location='cpu')
    print(" Cache file loaded successfully.")
else:
    raise FileNotFoundError(f" Cache Path incorrect: {cache_path}")

class VulnerabilityDataset(Dataset):
    def __init__(self, cache: dict, split: str):
        indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
        self.labels = self.cwe_labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.cwe_labels[idx]
        }

train_ds = VulnerabilityDataset(cache, split="train")
val_ds = VulnerabilityDataset(cache, split="val")

counts = np.bincount(train_ds.labels.numpy(), minlength=cfg['num_cwe_classes']).astype(float)
CLASS_WEIGHTS = torch.tensor(len(train_ds) / (cfg['num_cwe_classes'] * counts), dtype=torch.float).to(DEVICE)

sampler, _ = get_balanced_sampler(train_ds)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'])

print(f" DataLoaders ready.")
print(f" Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
print(f" CLASS_WEIGHTS: {CLASS_WEIGHTS}")

 Cache file loaded successfully.
 DataLoaders ready.
 Train samples: 12994 | Val samples: 1528
 CLASS_WEIGHTS: tensor([1.6762, 1.5166, 1.1727, 2.2159, 0.5737, 1.2194, 1.7693, 0.4326],
       device='cuda:0')


In [5]:
def validate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(ids, mask)
            logits = outputs['logits']
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    return macro_f1

In [ ]:
def train():
    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg['learning_rate']), weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, 100, len(train_loader)*cfg['epochs'])
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_f1 = 0.0
    weights_tensor = torch.ones(cfg['num_cwe_classes']).to(DEVICE)
    ckpt_path = f"{BASE_DIR}/{cfg['checkpoint_dir']}/best_model.pt"
    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
                model.load_state_dict(ckpt['model_state_dict'], strict=False)
                start_epoch = ckpt.get('epoch', -1) + 1
                best_f1 = ckpt.get('val_f1', 0.0)

                if 'class_weights' in ckpt:
                    weights_tensor = torch.tensor(ckpt['class_weights']).float().to(DEVICE)
                    print(" Successfully loaded weights and class_weights!")

                print(f" Resuming from Epoch {start_epoch} | Previous Best F1: {best_f1:.4f}")
            else:
                model.load_state_dict(ckpt, strict=False)
                print(" Weights loaded directly (Non-dict format)")
        except Exception as e:
            print(f" Note: {e}. Starting fresh.")

    # 3. Training Loop
    for epoch in range(start_epoch, cfg['epochs']):
        model.train()
        total_loss = 0

        for b in train_loader:
            ids = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbls = b['labels'].to(DEVICE)

            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=cfg['use_amp']):
                outputs = model(ids, mask)

                loss_rdrop = rdrop_criterion(outputs['logits'], outputs['logits'], lbls, weights_tensor)
                loss_scl = scl_criterion(outputs['features'], lbls)

                loss = loss_rdrop + (cfg['scl_weight'] * loss_scl)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total_loss += loss.item()

        f1 = validate(model, val_loader)
        avg_loss = total_loss/len(train_loader)
        print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | Val F1: {f1:.4f}")

        save_dict = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': f1,
            'class_weights': weights_tensor.cpu().numpy().tolist(),
            'optimizer_state_dict': opt.state_dict()
        }

        torch.save(save_dict, f"{BASE_DIR}/{cfg['checkpoint_dir']}/latest_v2.pt")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(save_dict, f"{BASE_DIR}/{cfg['checkpoint_dir']}/best_model_v2.pt")
            print(f" New Record Best F1 Updated: {best_f1:.4f}")


    baseline_f1 = 0.7420
    improvement = ((best_f1 - baseline_f1) / baseline_f1) * 100
    print(f"\n Training Complete Best F1: {best_f1:.4f} | Improvement vs Baseline: {improvement:.2f}%")


train()

 Resuming from Epoch 12 | Previous Best F1: 0.6923
Epoch 13 | Avg Loss: 0.3935 | Val F1: 0.6813
Epoch 14 | Avg Loss: 0.3357 | Val F1: 0.6830
